In [1]:
import pandas as pd
import arrow
import numpy as np
import matplotlib.pyplot as plt

# Data read-ins
gas_clean = pd.read_parquet("/Users/Ben.Pharris/Documents/project-dev/Media Measurement/data/clean/ga_cleaned.parquet")
gas_raw = pd.read_parquet("/Users/Ben.Pharris/Documents/project-dev/Media Measurement/data/cache/ga.parquet")
subpv = pd.read_csv("/Users/Ben.Pharris/Downloads/gas_subpv.csv")

spend = pd.read_parquet("/Users/Ben.Pharris/Documents/project-dev/Media Measurement/data/clean/spend_cleaned.parquet")

In [ ]:
#spend['date'] = pd.to_datetime(spend['date'])

In [ ]:
# Column Name Cleanup and Date restriction to Q4

dfs = {
        "gas_clean": gas_clean,
        "gas_raw": gas_raw,
        "subpv": subpv,
        "spend": spend
       }


for name, df in dfs.items():
        df.columns = df.columns.str.lower().str.replace(r'[.,\-_/ ]', '_', regex=True)
        df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')
        dfs[name] = df.loc[df["date"].between('2025-10-01', '2025-12-31', inclusive='both')]


for name, df in dfs.items():
    print(f"{name} Description: {df.describe()}")

globals().update(dfs)

gas_clean Description:                                 date           gas        pr_gas  \
count                          39358  39358.000000  39358.000000   
mean   2025-11-06 11:15:32.791300608     13.029956      0.017379   
min              2025-10-01 00:00:00   -389.000000    -10.000000   
25%              2025-10-19 00:00:00      1.000000      0.000000   
50%              2025-11-06 00:00:00      4.000000      0.000000   
75%              2025-11-25 00:00:00     11.000000      0.000000   
max              2025-12-14 00:00:00   1778.000000     35.000000   
std                              NaN     39.557917      0.553376   

       net_reporting_gas  
count       39358.000000  
mean           13.012577  
min          -389.000000  
25%             1.000000  
50%             4.000000  
75%            11.000000  
max          1778.000000  
std            39.559764  
gas_raw Description:                                 date           gas        pr_gas  \
count                          3

In [5]:
print(gas_raw['date'].min())

2024-07-01 00:00:00


In [20]:
#Total GA Comparison

print(f"Q4 RAW Total GAS : {gas_raw[(gas_raw['date'] >= '2025-10-01') & (gas_raw['date'] <= '2025-12-31')]['net_reporting_gas'].sum()}")
print(f"Q4 CLEAN Total GAS : {gas_clean[(gas_clean['date'] >= '2025-10-01') & (gas_clean['date'] <= '2025-12-31')]['net_reporting_gas'].sum()}")
print(f"Q4 SUBPV Total GAS : {subpv[(subpv['date'] >= '2025-10-01') & (subpv['date'] <= '2025-12-31')]['actuals'].sum()}")

Q4 RAW Total GAS : 512149
Q4 CLEAN Total GAS : 512149
Q4 SUBPV Total GAS : 512143.0


In [4]:
#Mapping Sub PV channels to sales channels
channel_map = {
    # Telesales
    'Direct': 'Telesales',
    'AXIOM-DIRECT': 'Telesales',
    'AXIOM-INDIRECT': 'Telesales',
    
    # Web
    'TikTok': 'Web',
    'WEB/DIGITAL': 'Web',
    
    # Indirect
    'Dealer Channel': 'Indirect',
    
    # Other
    'Alternate Distribution Channel': 'Other',
    'Dish Internal': 'Other',
    'Unknown': 'Other',
    'University Bookstore': 'Other',
    'Legacy': 'Other',
    'Small and Medium Business': 'Other'
}

# 2. Apply the map
# We use .fillna(subpv['Channel']) to keep values like 'Amazon' or 'Apple' that aren't in the dict
# We use .fillna('Other') at the end to catch any original Null/NaN values
subpv['channel'] = subpv['channel'].map(channel_map).fillna(subpv['channel']).fillna('Other')

# Check the results
print(subpv['channel'].value_counts(dropna=False))

channel
Other                               114
Amazon                               92
Direct Distribution Partner          92
Web Sales                            92
WalMart                              92
Target                               92
Apple                                92
National Retailer                    92
Telesales                            92
Cross Sell                           92
Best Buy                             92
Incomm                               75
Axiom Direct                         75
eTailer                              73
Axiom Indirect                       71
The Kroger Co.                       56
D&H Distribution                     32
Mobile Now HSN/QVC                   10
Sears Authorized Hometown Stores      1
Web                                   1
Name: count, dtype: int64


In [5]:
#Removing Geographic Dimension from GA data

gas_clean = gas_clean.groupby(['date', 'channel']).sum().reset_index()

gas_raw = gas_raw.groupby(['date', 'channel']).sum().reset_index()

In [6]:
print(f"Q4 SUBPV Total GAS : {subpv[(subpv['date'] >= '2025-10-01') & (subpv['date'] <= '2025-12-31')]['actuals'].sum()}")
print(f"Q4 Raw Total GAS : {gas_raw[(gas_raw['date'] >= '2025-10-01') & (gas_raw['date'] <= '2025-12-31')]['net_reporting_gas'].sum()}")
print(f"Q4 SQL Total GAS : {gas_raw[(gas_raw['date'] >= '2025-10-01') & (gas_raw['date'] <= '2025-12-31')]['net_reporting_gas'].sum()}")


Q4 SUBPV Total GAS : 512143.0
Q4 Raw Total GAS : 512149
Q4 SQL Total GAS : 512149


In [7]:
gas_raw_channels = set(gas_raw['channel'].unique())
subpv_channel = set(subpv['channel'].unique())

In [8]:
ga_diff = gas_raw_channels.difference(subpv_channel)
print(ga_diff)

{'National Retail', 'Indirect'}


In [9]:
other_diff = subpv_channel.difference(gas_raw_channels)
print(other_diff)

{'National Retailer', 'Sears Authorized Hometown Stores', 'Incomm', 'Mobile Now HSN/QVC', 'Target', 'Axiom Direct', 'The Kroger Co.', 'Direct Distribution Partner', 'Axiom Indirect', 'Best Buy', 'WalMart', 'eTailer', 'Web Sales', 'D&H Distribution'}


In [10]:
#Cleaning Up Funnel

print(f"Before Mapping Funnel: {spend['funnel'].unique()}")

funnel_map = {
    'Upper Funnel': 'Upper',
    'Middle': 'Upper',
    'Lower Funnel': 'Lower',
    'Lower': 'Lower',
    'Upper': 'Upper'
}

spend['funnel'] = spend['funnel'].map(funnel_map)

print(f"After Mapping Funnel: {spend['funnel'].unique()}")

Before Mapping Funnel: ['Upper Funnel' 'Lower Funnel' 'Lower' 'Upper' 'Middle']
After Mapping Funnel: ['Upper' 'Lower']


In [11]:
# Grouping Spend

spend = spend.groupby(['date', 'sales_channel', 'funnel'])['spend'].sum().reset_index()

In [12]:
spend_piv = spend.pivot_table(index=['date', 'sales_channel'], columns='funnel', values='spend', aggfunc='sum').reset_index()

In [13]:
total_uf_spend = spend_piv['Upper'].sum()

total_lf_spend = spend_piv['Lower'].sum()

print(f"Total Upper Funnel Spend: {total_uf_spend}")
print(f"Total Lower Funnel Spend: {total_lf_spend}")

Total Upper Funnel Spend: 333292355.611509
Total Lower Funnel Spend: 493758710.986241


In [15]:
# no decimals
print(f"{(total_lf_spend + total_uf_spend):,.0f}")

# or with two decimals and a dollar sign
print(f"${(total_lf_spend + total_uf_spend):,.2f}")

827,051,067
$827,051,066.60
